In [ ]:
import os
from typing import Annotated
import sqlite3

from dotenv import load_dotenv

from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
    trim_messages,
)
from langchain_groq import ChatGroq

from langgraph.graph import START, END, MessagesState, StateGraph
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages

load_dotenv()

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [ ]:
SYSTEM_PROMPT = SystemMessage(
    content="You are a helpful assistant."
)


class State(MessagesState):
    summary: str



In [ ]:
# ------------------------
# Nodes
# ------------------------

def summarize_node(state: State):
    if len(state["messages"]) < 8:
        return {}

    print("\n========== SUMMARY NODE ==========")

    old_summary = state.get("summary", "")

    summary_prompt = (
        f"Current summary:\n{old_summary}\n\n"
        if old_summary
        else ""
    )

    summary_prompt += (
        "Update the conversation summary using the messages below.\n"
        "Keep important facts and user preferences.\n"
        "Return only the summary."
    )

    response = model.invoke(
        [
            SystemMessage(content=summary_prompt),
            *state["messages"],
        ]
    )

    print("Conversation summarized.")

    keep_last = 4
    to_remove = [
        RemoveMessage(id=m.id)
        for m in state["messages"][:-keep_last]
    ]

    print(f"Deleted {len(to_remove)} old messages from graph state.")
    print("==================================\n")

    return {
        "summary": response.content,
        "messages": to_remove,
    }


def chat_node(state: State):
    print("\n========== CHAT NODE ==========")

    messages = state["messages"]

    if state.get("summary"):
        print("Using conversation summary.")
        messages = [
            SystemMessage(
                content=f"Conversation summary:\n{state['summary']}"
            )
        ] + messages

    trimmed = trim_messages(
        messages,
        max_tokens=400,
        strategy="last",
        token_counter=model,
        include_system=True,
        start_on="human",
    )

    if len(trimmed) != len(messages):
        print(
            f"Trimmed context from {len(messages)} messages to {len(trimmed)} messages."
        )
    else:
        print("No trimming needed.")

    response = model.invoke(trimmed)

    print("================================\n")

    return {
        "messages": [response]
    }


# ------------------------
# Router
# ------------------------

def router(state: State):
    if len(state["messages"]) >= 8:
        return "summarize"
    return "chat"


In [ ]:

# ------------------------
# Graph
# ------------------------

builder = StateGraph(State)

builder.add_node("chat", chat_node)
builder.add_node("summarize", summarize_node)

builder.add_conditional_edges(
    START,
    router,
)

builder.add_edge("summarize", "chat")
builder.add_edge("chat", END)

conn = sqlite3.connect("chatbot_STM.db", check_same_thread=False)

checkpointer = SqliteSaver(conn)

graph = builder.compile(checkpointer=checkpointer)


In [ ]:

config = {
    "configurable": {
        "thread_id": "demo-thread"
    }
}

print("=" * 60)
print("LangGraph Chatbot")
print("Type 'exit' to quit.")
print("=" * 60)

while True:
    user = input("\nYou: ")

    if user.lower() == "exit":
        break

    result = graph.invoke(
        {
            "messages": [HumanMessage(content=user)]
        },
        config=config,
    )
    
    print(f"User: {user}")
    print("\nAssistant:", result["messages"][-1].content)